# Ứng dụng Dự đoán Điểm thi (Hồi quy tuyến tính)
**Cảnh báo:** Dữ liệu trong bài là dữ liệu tổng hợp phục vụ giảng dạy. Không sử dụng mô hình này để đánh giá sinh viên thật.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_names = artifact["feature_names"]

In [2]:
hours = widgets.FloatSlider(value=6, min=0, max=20, step=.5, description="Gio hoc:", continuous_update=False)
attendance = widgets.FloatSlider(value=80, min=50, max=100, step=1, description="Chuyen can:", continuous_update=False)
assignments = widgets.IntSlider(value=7, min=0, max=10, step=1, description="Bai tap:", continuous_update=False)
previous = widgets.FloatSlider(value=6.5, min=0, max=10, step=.1, description="Diem truoc:", continuous_update=False)
predict_button = widgets.Button(description="Du doan", button_style="primary")
output = widgets.Output()

def input_frame():
    values = {
        "hours_studied": hours.value,
        "attendance": attendance.value,
        "assignments_completed": assignments.value,
        "previous_score": previous.value,
    }
    return pd.DataFrame([[values[name] for name in feature_names]], columns=feature_names)

def on_predict(_):
    predict_button.disabled = True
    with output:
        output.clear_output()
        try:
            raw = float(model.predict(input_frame())[0])
            print(f"Diem thi du doan: {np.clip(raw, 0, 10):.2f} /10")
        except Exception:
            print("Khong the du doan. Hay kiem tra du lieu va artifact.")
        finally:
            predict_button.disabled = False

predict_button.on_click(on_predict)
display(widgets.VBox([hours, attendance, assignments, previous, predict_button, output]))